# ZIP — Treino ShanghaiTech Part A (Kaggle)

**Antes de correr:**
1. Adiciona o dataset em *Add data* (upload do teu zip, ou dataset público ShanghaiTech)
2. Activa GPU: *Session options → Accelerator → GPU T4 x2 ou P100*
3. Corre as células em ordem

**Em cada nova sessão:** corre apenas as células 1, 3 e 4 (dataset não precisa de ser refeito se `/kaggle/working/ZIP/data/sha/` já existir)

## Célula 1 — Setup: repo + dependências

In [ ]:
import os, subprocess, sys, glob

# ── Paths ────────────────────────────────────────────────
WORK_DIR = '/kaggle/working'
REPO_DIR = f'{WORK_DIR}/ZIP'
CKPT_DIR = f'{WORK_DIR}/checkpoints/sha_official'

# ── Clona o repo ─────────────────────────────────────────
if not os.path.exists(f'{REPO_DIR}/trainer.py'):
    print('📥 A clonar repositório ZIP...')
    # Remove pasta incompleta se existir
    if os.path.exists(REPO_DIR):
        import shutil; shutil.rmtree(REPO_DIR)
    result = subprocess.run(
        ['git', 'clone', 'https://github.com/Yiming-M/ZIP.git', REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('Falha ao clonar o repositório!')
    print('✅ Repositório clonado')
else:
    print(f'✅ Repositório já existe em {REPO_DIR}')

# Confirma que trainer.py existe
assert os.path.exists(f'{REPO_DIR}/trainer.py'), \
    f'trainer.py não encontrado em {REPO_DIR}! Verifica o clone.'
print(f'✅ trainer.py confirmado: {REPO_DIR}/trainer.py')

# ── Mostra estrutura do repo ─────────────────────────────
print('\n📂 Estrutura do repo:')
for f in sorted(glob.glob(f'{REPO_DIR}/*.py') + glob.glob(f'{REPO_DIR}/*.sh') + glob.glob(f'{REPO_DIR}/*.txt')):
    print(f'   {os.path.basename(f)}')

# ── Dependências ─────────────────────────────────────────
os.chdir(REPO_DIR)
print('\n📦 A instalar dependências...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scipy', 'einops'], check=True)
if os.path.exists('requirements.txt'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('✅ Dependências instaladas')

# ── GPU ──────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)}')
else:
    print('⚠️  SEM GPU — activa em Session options → Accelerator')

## Célula 2 — Diagnóstico do dataset (corre se tiveres dúvidas)

In [ ]:
import glob, os

print('=== Datasets disponíveis em /kaggle/input/ ===')
input_dirs = glob.glob('/kaggle/input/*')
if not input_dirs:
    print('❌ Nenhum dataset adicionado!')
    print('   → Clica em Add data e adiciona o teu dataset ShanghaiTech')
else:
    for d in input_dirs:
        print(f'  {d}')
    print()
    # Procura part_A
    hits = glob.glob('/kaggle/input/**/part_A', recursive=True)
    if hits:
        SRC = hits[0]
        print(f'✅ part_A encontrado: {SRC}')
        ti = glob.glob(f'{SRC}/train_data/images/*.jpg')
        vi = glob.glob(f'{SRC}/test_data/images/*.jpg')
        tm = glob.glob(f'{SRC}/train_data/ground-truth/*.mat')
        print(f'   train: {len(ti)} imagens, {len(tm)} labels')
        print(f'   test : {len(vi)} imagens')
    else:
        print('❌ part_A não encontrado. Estrutura actual:')
        os.system('find /kaggle/input -maxdepth 5 -type d')

print('\n=== Estrutura sha/ já preparada? ===')
ti2 = glob.glob('/kaggle/working/ZIP/data/sha/train/images/*.jpg')
vi2 = glob.glob('/kaggle/working/ZIP/data/sha/val/images/*.jpg')
if ti2:
    print(f'✅ train: {len(ti2)} imgs | val: {len(vi2)} imgs — podes saltar a Célula 3')
else:
    print('ℹ️  Ainda não preparado — corre a Célula 3')

## Célula 3 — Preparação do dataset (só na 1.ª sessão)

In [ ]:
import os, glob, shutil, zipfile
import numpy as np
import scipy.io as sio

REPO_DIR        = '/kaggle/working/ZIP'
FINAL_TRAIN_IMG = f'{REPO_DIR}/data/sha/train/images'
FINAL_TRAIN_LBL = f'{REPO_DIR}/data/sha/train/labels'
FINAL_VAL_IMG   = f'{REPO_DIR}/data/sha/val/images'
FINAL_VAL_LBL   = f'{REPO_DIR}/data/sha/val/labels'

# Já está pronto?
if len(glob.glob(f'{FINAL_TRAIN_IMG}/*.jpg')) >= 300:
    print('✅ Dataset já preparado — salta esta célula')
else:
    # ── Localiza part_A ──────────────────────────────────
    hits = glob.glob('/kaggle/input/**/part_A', recursive=True)

    # Se não veio via Add data, tenta encontrar um zip e extrair
    if not hits:
        zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
        if zips:
            print(f'📂 A extrair zip: {zips[0]}')
            with zipfile.ZipFile(zips[0], 'r') as z:
                z.extractall('/kaggle/working/extracted')
            hits = glob.glob('/kaggle/working/extracted/**/part_A', recursive=True)

    if not hits:
        print('❌ part_A não encontrado!')
        print('   Corre a Célula 2 para diagnosticar.')
        raise FileNotFoundError('part_A não encontrado em /kaggle/input/')

    SRC = hits[0]
    print(f'✅ Fonte: {SRC}')

    for d in [FINAL_TRAIN_IMG, FINAL_TRAIN_LBL, FINAL_VAL_IMG, FINAL_VAL_LBL]:
        os.makedirs(d, exist_ok=True)

    def convert_split(split, img_dst, lbl_dst):
        imgs = sorted(glob.glob(f'{SRC}/{split}_data/images/*.jpg'))
        mats = sorted(glob.glob(f'{SRC}/{split}_data/ground-truth/*.mat'))
        for img in imgs:
            fname = os.path.basename(img).replace('IMG_', '')
            shutil.copy(img, os.path.join(img_dst, fname))
        for mat_path in mats:
            mat = sio.loadmat(mat_path)
            try:
                pts = mat['image_info'][0][0][0][0][0].astype(np.float32)
            except Exception:
                pts = np.zeros((0, 2), dtype=np.float32)
            img_id = os.path.basename(mat_path).replace('GT_IMG_', '').replace('.mat', '')
            np.save(os.path.join(lbl_dst, f'{img_id}.npy'), pts)
        return len(imgs), len(mats)

    ti, tl = convert_split('train', FINAL_TRAIN_IMG, FINAL_TRAIN_LBL)
    vi, vl = convert_split('test',  FINAL_VAL_IMG,   FINAL_VAL_LBL)
    print(f'✅ train: {ti} imgs / {tl} labels | val: {vi} imgs / {vl} labels')

## Célula 4 — Treino (retoma automaticamente se já houver checkpoint)

In [ ]:
import os, glob, subprocess, sys

REPO_DIR = '/kaggle/working/ZIP'
CKPT_DIR = '/kaggle/working/checkpoints/sha_official'
TRAINER  = f'{REPO_DIR}/trainer.py'

os.makedirs(CKPT_DIR, exist_ok=True)
os.chdir(REPO_DIR)

# ── Valida trainer.py ─────────────────────────────────────
assert os.path.exists(TRAINER), (
    f'trainer.py não encontrado em {REPO_DIR}!\n'
    'Corre a Célula 1 primeiro.'
)

# ── Valida dataset ───────────────────────────────────────
n_train = len(glob.glob(f'{REPO_DIR}/data/sha/train/images/*.jpg'))
assert n_train >= 300, (
    f'Dataset incompleto ({n_train} imagens de treino)!\n'
    'Corre a Célula 3 primeiro.'
)
print(f'✅ Dataset OK: {n_train} imagens de treino')

# ── Checkpoint para retomar ───────────────────────────────
ckpts = sorted(
    glob.glob(f'{CKPT_DIR}/epoch_*.pt') + glob.glob(f'{CKPT_DIR}/epoch_*.pth'),
    key=lambda p: int(''.join(filter(str.isdigit,
        os.path.basename(p).split('epoch_')[-1].split('.')[0])) or '0')
)
resume_args = []
if ckpts:
    last = ckpts[-1]
    epoch_num = os.path.basename(last)
    print(f'▶️  A retomar de: {epoch_num}')
    help_txt = subprocess.run([sys.executable, TRAINER, '--help'],
                              capture_output=True, text=True).stdout
    flag = '--resume' if 'resume' in help_txt else '--ckpt'
    resume_args = [flag, last]
else:
    print('🆕 Nenhum checkpoint — a começar do zero')

# ── Comando de treino ─────────────────────────────────────
cmd = [
    sys.executable, TRAINER,          # caminho absoluto — sem ambiguidade
    '--model_name',            'ebc_s',
    '--dataset',               'sha',
    '--input_size',            '224',
    '--num_crops',             '2',
    '--batch_size',            '8',
    '--optimizer',             'adam',
    '--lr',                    '1e-4',
    '--backbone_lr',           '1e-4',
    '--weight_decay',          '1e-4',
    '--backbone_weight_decay', '1e-4',
    '--scheduler',             'cos_restarts',
    '--warmup_epochs',         '25',
    '--warmup_lr',             '1e-5',
    '--eta_min',               '1e-6',
    '--T_0',                   '5',
    '--T_mult',                '2',
    '--total_epochs',          '1300',
    '--eval_start',            '100',
    '--eval_freq',             '0.25',
    '--save_freq',             '50',
    '--save_best_k',           '5',
    '--aug_min_scale',         '0.75',
    '--aug_max_scale',         '2.0',
    '--block_size',            '8',
    '--reg_loss',              'zipnll',
    '--aux_loss',              'msmae',
    '--weight_cls',            '1.0',
    '--weight_reg',            '1.0',
    '--weight_aux',            '0.025',
    '--scales',                '1', '2', '4',
    '--amp',
    '--num_workers',           '4',
    '--ckpt_dir_name',         'sha_official',
    '--ckpt_dir',              CKPT_DIR,
] + resume_args

print('\n🚀 A iniciar treino...')
print('   Trainer :', TRAINER)
print('   Dataset :', f'{REPO_DIR}/data/sha/')
print('   CKPT    :', CKPT_DIR)
print('='*60 + '\n')

try:
    subprocess.run(cmd, check=True)
    print('\n✅ Treino concluído!')
except subprocess.CalledProcessError as e:
    print(f'\n❌ Erro (exit code {e.returncode})')
    print('   Corre a Célula 2 para diagnosticar o dataset')
    print('   Verifica se a Célula 1 correu sem erros')
except KeyboardInterrupt:
    print('\n⚠️  Interrompido. Na próxima sessão corre células 1 e 4.')

## Célula 5 — Ver resultados e checkpoints

In [ ]:
import glob, os, json

CKPT_DIR = '/kaggle/working/checkpoints/sha_official'

for log in [f'{CKPT_DIR}/results.json', f'{CKPT_DIR}/log.json']:
    if os.path.exists(log):
        with open(log) as f:
            print(json.dumps(json.load(f), indent=2))
        break

ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.pt') + glob.glob(f'{CKPT_DIR}/*.pth'))
print(f'\n📂 {len(ckpts)} checkpoints:')
for c in ckpts[-8:]:
    mb = os.path.getsize(c) / 1e6
    print(f'   {os.path.basename(c):50s}  {mb:.0f} MB')
print('\n💡 File → Save version → Output files para guardar permanentemente')